In [ ]:
import pandas as pd
import glob

# path to your files
files = glob.glob("*.tsv")  # <-- change if needed

dfs = []

for f in files:
    df = pd.read_csv(f, sep="\t", header=None)
    
    df = df.iloc[:, :7]  # keep only needed columns
    
    df.columns = [
        "sample_contig",   # e.g. SRR..._Contig-1
        "contig_length",   # nt
        "evalue",
        "pident",
        "align_length",
        "subject",
        "annotation"
    ]
    
    # extract virus name from annotation
    df["virus"] = df["annotation"].str.extract(r"\[(.*?)\]")
    
    # extract sample name
    df["sample"] = df["sample_contig"].str.split("_cap3_").str[0]
    
    dfs.append(df)

df_all = pd.concat(dfs, ignore_index=True)

print("Total rows:", len(df_all))
df_all.head()

In [ ]:
import numpy as np

# replace zero e-values (avoid log issues)
df_all["evalue"] = df_all["evalue"].replace(0, 1e-300)

# filtering thresholds (adjust if needed)
df_filtered = df_all[
    (df_all["evalue"] <= 1e-5) &
    (df_all["pident"] >= 30) &
    (df_all["contig_length"] >= 300)
].copy()

print("After filtering:", len(df_filtered))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(12,6))

sns.boxplot(
    data=df_filtered,
    x="sample",
    y="contig_length"
)

plt.xticks(rotation=45, ha="right")
plt.ylabel("Contig length (nt)")
plt.xlabel("Sample")
plt.title("Contig length distribution per sample")

plt.show()